# Prediction intervals with MMF
This notebook shows how to get native prediction intervals from MMF, using daily sales of a sample of [M5 (Walmart)](https://www.sciencedirect.com/science/article/pii/S0169207021001874) items.

Models with native intervals (here `AutoETS` and `SeasonalNaive`) return their quantiles in the backtest and at scoring. The intervals are the models' own bands, **not calibrated**, so the notebook measures their coverage on the backtest. Models without native intervals (here `CrostonSBA`) get NULL interval columns.

See the [Prediction Intervals](https://github.com/databricks-industry-solutions/many-model-forecasting#prediction-intervals) section of the README for the settings.

### Cluster setup

Use a CPU cluster with [Databricks Runtime 18 for ML](https://docs.databricks.com/aws/en/release-notes/runtime/18ml), or serverless compute.

### Install and import packages

In [ ]:
%pip install -r ../../requirements-local.txt --quiet
%pip install datasetsforecast==0.0.8 matplotlib --quiet
dbutils.library.restartPython()

In [ ]:
import numpy as np
import pandas as pd
from datasetsforecast.m5 import M5
from mmf_sa import run_forecast

### Prepare data
We download M5 and keep a random sample of item x store series. The full long table does not fit in a small driver, so we sample the wide CSV before melting it. Leading zeros (before an item's launch) are dropped.

In [ ]:
catalog = "mmf"  # Name of the catalog we use to manage our assets
db = "m5"  # Name of the schema we use to manage our assets (e.g. datasets)
user = spark.sql("select current_user() as user").collect()[0]["user"]
n = 500  # Number of series

_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

In [ ]:
M5.download(directory="/tmp/m5")
path = "/tmp/m5/m5/datasets"
calendar = pd.read_csv(f"{path}/calendar.csv", usecols=["date"], parse_dates=["date"])
calendar["d"] = [f"d_{i + 1}" for i in range(len(calendar))]
wide = pd.read_csv(f"{path}/sales_train_evaluation.csv")
wide["unique_id"] = wide["item_id"] + "_" + wide["store_id"]
wide = wide.sample(n=n, random_state=7)
day_cols = [c for c in wide.columns if c.startswith("d_")]
train = wide.melt(id_vars=["unique_id"], value_vars=day_cols, var_name="d", value_name="y")
train = train.merge(calendar, on="d").rename(columns={"date": "ds"}).drop(columns="d")
train["y"] = train["y"].astype(float)
train = train.sort_values(["unique_id", "ds"])
train = train[train.groupby("unique_id")["y"].cumsum() > 0]  # drop leading zeros

(
    spark.createDataFrame(train)
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.daily_train_pi")
)
print(f"{train['unique_id'].nunique()} series, {train['ds'].min().date()} to {train['ds'].max().date()}")

### Run MMF with prediction intervals
`prediction_intervals` is the only new argument. We ask for 80% and 95% intervals, both as a quantile map and as bound columns (`output_format: both`).

The backtest has 3 windows of 28 days (`backtest_length=84`, `stride=28`); its forecasts carry the native quantiles too, which is what we use below to measure the coverage of the bands.

In [ ]:
active_models = [
    "StatsForecastBaselineSeasonalNaive",
    "StatsForecastAutoETS",
    "StatsForecastCrostonSBA",
]

run_id = run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.daily_train_pi",
    scoring_data=f"{catalog}.{db}.daily_train_pi",
    scoring_output=f"{catalog}.{db}.daily_pi_scoring_output",
    evaluation_output=f"{catalog}.{db}.daily_pi_evaluation_output",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="D",
    prediction_length=28,
    backtest_length=84,
    stride=28,
    metric="smape",
    train_predict_ratio=1,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/m5_prediction_intervals",
    use_case_name="m5_daily_pi",
    prediction_intervals={"enabled": True, "levels": [80, 95], "output_format": "both"},
)

### Forecasts with intervals
`scoring_output` now has `y_quantiles` (a map from quantile level to an array over the horizon) and the bound columns `y_lo_80`, `y_hi_80`, `y_lo_95`, `y_hi_95`. They are NULL for `CrostonSBA`, which has no native intervals.

In [ ]:
display(spark.sql(f"""
    SELECT unique_id, model, ds, y, y_lo_80, y_hi_80, y_lo_95, y_hi_95
    FROM {catalog}.{db}.daily_pi_scoring_output
    WHERE run_id = '{run_id}'
    ORDER BY unique_id, model
    LIMIT 9
"""))

In [ ]:
import matplotlib.pyplot as plt

# A few series, for every model.
series = [r["unique_id"] for r in spark.sql(f"""
    SELECT DISTINCT unique_id FROM {catalog}.{db}.daily_pi_scoring_output
    WHERE run_id = '{run_id}'
    ORDER BY unique_id
    LIMIT 3
""").collect()]
fc = spark.sql(f"""
    SELECT unique_id, model, ds, y, y_lo_80, y_hi_80, y_lo_95, y_hi_95
    FROM {catalog}.{db}.daily_pi_scoring_output
    WHERE run_id = '{run_id}' AND unique_id IN ({", ".join(f"'{s}'" for s in series)})
""").toPandas()

fig, axes = plt.subplots(len(series), len(active_models), figsize=(6 * len(active_models), 3 * len(series)),
                         sharex=True, squeeze=False)
for i, sid in enumerate(series):
    history = train[train["unique_id"] == sid].tail(120)
    for j, model in enumerate(active_models):
        ax = axes[i, j]
        ax.plot(history["ds"], history["y"], color="black", lw=1, label="history")
        rows = fc[(fc["unique_id"] == sid) & (fc["model"] == model)]
        if len(rows):
            r = rows.iloc[0]
            for level, alpha in ((95, 0.2), (80, 0.4)):
                if r[f"y_lo_{level}"] is not None:  # NULL for models without native intervals
                    ax.fill_between(r["ds"], r[f"y_lo_{level}"], r[f"y_hi_{level}"], color="tab:blue", alpha=alpha,
                                    label=f"{level}%")
            ax.plot(r["ds"], r["y"], color="tab:blue", lw=1.5, label="forecast")
        ax.set_title(f"{sid} - {model.removeprefix('StatsForecast')}", fontsize=10)
axes[0, 0].legend(loc="upper left")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### How well do the native bands cover?
The backtest forecasts are out of sample: each window is forecast from the history before it. So the share of backtest actuals inside each native band measures its coverage, to compare with the nominal 0.80 and 0.95. The bands are not calibrated, so a gap is expected.

In [ ]:
display(spark.sql(f"""
    WITH points AS (
      SELECT model, a,
             try_element_at(forecast_quantiles, '0.1')[k] AS lo_80, try_element_at(forecast_quantiles, '0.9')[k] AS hi_80,
             try_element_at(forecast_quantiles, '0.025')[k] AS lo_95, try_element_at(forecast_quantiles, '0.975')[k] AS hi_95
      FROM {catalog}.{db}.daily_pi_evaluation_output
      LATERAL VIEW posexplode(actual) p AS k, a
      WHERE run_id = '{run_id}' AND forecast_quantiles IS NOT NULL AND metric_name = 'smape'
    )
    SELECT model,
           ROUND(AVG(CASE WHEN lo_80 IS NULL OR hi_80 IS NULL THEN NULL
                          WHEN a BETWEEN lo_80 AND hi_80 THEN 1.0 ELSE 0.0 END), 3) AS coverage_80,
           ROUND(AVG(CASE WHEN lo_95 IS NULL OR hi_95 IS NULL THEN NULL
                          WHEN a BETWEEN lo_95 AND hi_95 THEN 1.0 ELSE 0.0 END), 3) AS coverage_95,
           COUNT(*) AS points
    FROM points
    GROUP BY model
    ORDER BY model
"""))

### Delete tables

In [ ]:
#display(spark.sql(f"DROP TABLE IF EXISTS {catalog}.{db}.daily_pi_scoring_output"))
#display(spark.sql(f"DROP TABLE IF EXISTS {catalog}.{db}.daily_pi_evaluation_output"))